## Evaluation of Mistral-7B on Question Answering

In this experiment, we evaluated the performance of **Mistral-7B-Instruct-v0.3** on a Question Answering (QA) task, following the same experimental protocol used for the other large language models in the project.

### Experimental Setup

- **Model**: Mistral-7B-Instruct-v0.3  
- **Task**: Question Answering (QA)  
- **Language**: English  
- **Pipeline**: Vanilla  
- **Number of examples**: 972  
- **Input data**:  
  - Each example consists of a sentence (context) and a list of automatically generated questions.
  - Questions were produced by the same Question Generation (QG) pipeline used for all other models, ensuring a fair comparison.

### QA Generation

Mistral-7B was prompted using a structured instruction-based prompt designed to:
- Condition the model on the given sentence.
- Answer all associated questions.
- Output only the list of answers, without explanations or additional text.

The model was used in a **zero-shot setting**, without any fine-tuning or retrieval augmentation.  
All answers were generated autoregressively using greedy decoding.

### Evaluation Method

To assess the semantic similarity of the generated answers, we employed **SBERT-based semantic similarity**:
- Each Mistral-generated answer was compared against the corresponding answers produced by the other baseline models.
- Sentence embeddings were computed using a pretrained SBERT model.
- Cosine similarity was used as the evaluation metric.
- Final scores were obtained by averaging similarity values across all evaluated examples.

### Purpose of the Experiment

This experiment aims to analyze how a **decoder-only instruction-tuned model**, such as Mistral-7B, behaves in a QA setting when:
- The task requires concise, context-grounded answers.
- No QA-specific fine-tuning or architectural adaptation is applied.

The results are used to compare Mistral-7B against other models under identical conditions, highlighting strengths and limitations related to model architecture and training objectives.


In [ ]:
!pip -q install transformers accelerate bitsandbytes sentencepiece rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
# all the import here
import sys
import os
import torch
import json
import argparse
from transformers import AutoTokenizer, AutoModelForCausalLM

from tqdm import tqdm
import csv
import numpy as np
from sentence_transformers import SentenceTransformer, util
import random
import ast
import re
import string
from collections import Counter
from rouge_score import rouge_scorer
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from QA.code.prompt import qa_prompt
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output_path", type=str, required=True)
    parser.add_argument("--sentence_type", type=str, required=True)
    parser.add_argument("--pipeline", type=str, required=True)
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
        load_in_4bit=True
    )

    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id
    model.eval()

    os.makedirs(os.path.dirname(args.output_path), exist_ok=True)

    # questions are from llama-8b
    qg_file = f"QG/llama-8b/{args.pipeline}_llama-8b.jsonl"

    with open(qg_file, "r", encoding="utf-8") as f:
        total_examples = sum(1 for _ in f)

    with open(qg_file, "r", encoding="utf-8") as f_in, \
         open(args.output_path, "w", encoding="utf-8") as f_out:

        for idx, line in enumerate(
            tqdm(f_in, total=total_examples, desc="Mistral QA", unit="ex")
        ):
            data = json.loads(line)

            sentence = data.get(args.sentence_type)
            questions = data.get("questions")

            if not sentence or not questions:
                continue

            prompt = (
                qa_prompt
                .replace("{{sentence}}", sentence)
                .replace("{{questions}}", questions)
            )

            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=64,
                    do_sample=False,
                )

            generated = tokenizer.decode(
                outputs[0][inputs.input_ids.shape[-1]:],
                skip_special_tokens=True
            ).strip()

            data["answers"] = generated
            f_out.write(json.dumps(data, ensure_ascii=False) + "\n")


if __name__ == "__main__":
    main()

Writing QA/code/Off-mistral-7b.py


In [ ]:
# Vanilla
!python -u QA/code/Off-mistral-7b.py \
  --output_path QA/mistral-7b/en/Off-en-vanilla.jsonl \
  --sentence_type en \
  --pipeline vanilla

tokenizer_config.json: 141kB [00:00, 85.6MB/s]
tokenizer.model: 100% 587k/587k [00:01<00:00, 318kB/s]
tokenizer.json: 1.96MB [00:00, 163MB/s]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 4.21MB/s]
config.json: 100% 601/601 [00:00<00:00, 5.10MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-12 08:43:49.578044: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-12 08:43:49.594467: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768207429.613696    1956 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:17682

In [ ]:
# Atomic
!python -u QA/code/Off-mistral-7b.py \
  --output_path QA/mistral-7b/en/Off-en-atomic.jsonl \
  --sentence_type en \
  --pipeline atomic

tokenizer_config.json: 141kB [00:00, 215MB/s]
tokenizer.model: 100% 587k/587k [00:01<00:00, 301kB/s]
tokenizer.json: 1.96MB [00:00, 103MB/s]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 3.87MB/s]
config.json: 100% 601/601 [00:00<00:00, 6.83MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-19 17:31:18.170339: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-19 17:31:18.187158: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768843878.205972    2113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:176884

In [ ]:
# Semantic
!python -u QA/code/Off-mistral-7b.py \
  --output_path QA/mistral-7b/en/Off-en-semantic.jsonl \
  --sentence_type en \
  --pipeline semantic

`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-19 18:14:48.780360: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-19 18:14:48.796616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768846488.815836   13481 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768846488.822444   13481 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768846488.839139   13481 computation_placer.cc:177] computation placer already r

---

# Evaluation

## SBERT

In [ ]:
# evaluation Vanilla
!python evaluation/sbert/off-sbert-compare-all.py \
  --reference_file QA/mistral-7b/en/Off-en-vanilla.jsonl \
  --pipeline vanilla

2026-01-19 19:15:04.034615: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768850104.055962   29132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768850104.062413   29132 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768850104.078459   29132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768850104.078494   29132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768850104.078497   29132 computation_placer.cc:177] computation placer alr

In [ ]:
# evaluation Atomic
!python evaluation/sbert/off-sbert-compare-all.py \
  --reference_file QA/mistral-7b/en/Off-en-atomic.jsonl \
  --pipeline atomic

2026-01-19 19:16:11.520213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768850171.541174   29502 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768850171.547656   29502 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768850171.564087   29502 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768850171.564114   29502 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768850171.564117   29502 computation_placer.cc:177] computation placer alr

In [ ]:
# evaluation Semantic
!python evaluation/sbert/off-sbert-compare-all.py \
  --reference_file QA/mistral-7b/en/Off-en-semantic.jsonl \
  --pipeline semantic

2026-01-19 19:18:48.598012: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768850328.619051   30284 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768850328.625520   30284 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768850328.641627   30284 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768850328.641669   30284 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768850328.641672   30284 computation_placer.cc:177] computation placer alr

## Comparison between LlaMA-8b and Mistral-7b

In [ ]:
# FILE PATH
llama_file = "QA/llama-8b/en-vanilla.jsonl"
mistral_file = "QA/mistral-7b/en/Off-en-vanilla.jsonl"

# LOAD FILES
def load_jsonl(path):
    data = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            data[obj["id"]] = obj
    return data

llama_data = load_jsonl(llama_file)
mistral_data = load_jsonl(mistral_file)

# IDS that are in commons
common_ids = list(set(llama_data.keys()) & set(mistral_data.keys()))
print(f"Found {len(common_ids)} common examples")

# we take only 5 examples
sample_ids = random.sample(common_ids, 5)

print("\n===== QUALITATIVE COMPARISON (5 EXAMPLES) =====\n")

def parse_list_field(field_value):
    """
    Tries to parse a string representation of a list into an actual list.
    Returns the list if successful, otherwise a list containing the original string.
    """
    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        cleaned = field_value.strip()
        # Heuristic: starts with [ and ends with ] might be a list
        if cleaned.startswith('[') and cleaned.endswith(']'):
            try:
                # Try standard JSON
                return json.loads(cleaned)
            except json.JSONDecodeError:
                try:
                    # Try Python literal syntax (handles single quotes)
                    return ast.literal_eval(cleaned)
                except:
                    pass
        return [field_value] # Return as single item if not parseable

    return []

for ex_id in sample_ids:
    llama = llama_data[ex_id]
    mistral = mistral_data[ex_id]

    print(f"🆔 ID: {ex_id}")
    print("-" * 60)

    print("📘 CONTEXT:")
    print(llama.get("en", "").strip())
    print()

    print("❓ QUESTIONS:")
    qs = parse_list_field(llama.get("questions", []))
    for q in qs:
        print(f"- {q}")
    print()

    print("🦙 LLaMA ANSWERS:")
    ans_l = parse_list_field(llama.get("answers", []))
    for a in ans_l:
        print(f"- {a}")
    print()

    print("🌀 MISTRAL ANSWERS:")
    ans_m = parse_list_field(mistral.get("answers", []))
    for a in ans_m:
        print(f"- {a}")
    print("\n" + "=" * 60 + "\n")

Found 971 common examples

===== QUALITATIVE COMPARISON (5 EXAMPLES) =====

🆔 ID: Wikivoyage_1:2381
------------------------------------------------------------
📘 CONTEXT:
The risk of getting stuck in the connection city is higher than usual right now, due to delays for screening and testing as well as extensive cancellations.

❓ QUESTIONS:
- What is the connection city?
- What is higher than usual right now?
- Why is the risk higher than usual right now?
- What are the reasons for the delays?
- What is the impact of cancellations on the risk?

🦙 LLaMA ANSWERS:
- the connection city
- the risk
- delays for screening and testing as well as extensive cancellations
- delays for screening and testing and extensive cancellations
- extensive cancellations

🌀 MISTRAL ANSWERS:
- The connection city
- The risk of getting stuck
- Due to delays for screening and testing as well as extensive cancellations
- Delays for screening and testing
- The impact of cancellations is an increase in the risk



In [ ]:

def calculate_metrics(original, predicted):

    original = original
    predicted = predicted

    # --- 1. Exact Match (EM) ---
    em = 1 if original == predicted else 0

    # --- 2. F1-Score (token overlap with frequency) ---
    orig_tokens = original.split()
    pred_tokens = predicted.split()

    common = Counter(orig_tokens) & Counter(pred_tokens)
    num_same = sum(common.values())

    if len(orig_tokens) == 0 or len(pred_tokens) == 0:
        f1 = int(orig_tokens == pred_tokens)
    elif num_same == 0:
        f1 = 0
    else:
        precision = num_same / len(pred_tokens)
        recall = num_same / len(orig_tokens)
        f1 = 2 * precision * recall / (precision + recall)

    return em, f1

for ex_id in common_ids:

    llama_answers = llama_data[ex_id].get("answers", [])
    mistral_answers = mistral_data[ex_id].get("answers", [])

    if isinstance(llama_answers, str):
        llama_answers = [llama_answers]
    if isinstance(mistral_answers, str):
        mistral_answers = [mistral_answers]

    for ref, pred in zip(llama_answers, mistral_answers):

        ref = str(ref)
        pred = str(pred)

        # EM + F1
        em, f1 = calculate_metrics(ref, pred)
        em_scores.append(em)
        f1_scores.append(f1)

        # ROUGE-L
        rl_score = scorer.score(ref, pred)['rougeL'].fmeasure
        rouge_scores.append(rl_score)

# =========================
print("\n📈 REPORT METRICHE DI CONFRONTO (VANILLA)")
print("-" * 50)
print(f"✅ Exact Match (EM):    {np.mean(em_scores):.4f}")
print(f"🎯 F1-Score (Overlap): {np.mean(f1_scores):.4f}")
print(f"📝 ROUGE-L:            {np.mean(rouge_scores):.4f}")
print("-" * 50)
print(f"Totale answer pairs: {len(em_scores)}")

Found 971 common examples

📈 REPORT METRICHE DI CONFRONTO (VANILLA)
--------------------------------------------------
✅ Exact Match (EM):    0.0672
🎯 F1-Score (Overlap): 0.5751
📝 ROUGE-L:            0.6927
--------------------------------------------------
Totale answer pairs: 3884
